# Chapter 13 — How LLMs Are Trained## The Token-as-Action MDP Mapping[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**No GPU required. Runtime: ~20 minutes.**Makes the token-as-action MDP mapping from Chapter 13 concrete and verifiable.

In [ ]:
import torchimport numpy as npimport matplotlib.pyplot as plttry:    from transformers import AutoTokenizer, AutoModelForCausalLM    print('Transformers available')except:    print('pip install transformers')

## 13.1 Tokens as Actions: The Complete MDPFor an LLM being trained with RLHF, every token generation step is an MDP action.

In [ ]:
# Demonstrate the MDP structure concretelyPROMPT = 'The capital of France is'RESPONSE_TOKENS = [' Paris', ',', ' which', ' is', ' a', ' major', ' city', '.', '<EOS>']print('=== Token-as-Action MDP ===\n')print(f'Initial state s_0: "{PROMPT}"')print()for i, token in enumerate(RESPONSE_TOKENS):    is_final = token == '<EOS>'    print(f'Step t={i+1}:')    print(f'  Action a_{i}: generate token "{token}"')    context = PROMPT + ''.join(RESPONSE_TOKENS[:i+1])    print(f'  New state s_{i+1}: "{context[:60]}..."' if len(context)>60 else f'  New state s_{i+1}: "{context}"')    if is_final:        print(f'  Reward r_{i+1}: r_φ(prompt, full_response)  ← reward model scores the complete response')    else:        print(f'  Reward r_{i+1}: -β·KL_t (KL divergence penalty for this token)')    print()

## 13.2 Computing Log-Probabilities of Tokens

In [ ]:
try:    model_name = 'distilgpt2'    tokenizer = AutoTokenizer.from_pretrained(model_name)    model = AutoModelForCausalLM.from_pretrained(model_name)    model.eval()        prompt = 'The capital of France is'    prompt_ids = tokenizer(prompt, return_tensors='pt')['input_ids']        # Full sequence: prompt + response    full_text = prompt + ' Paris, which is a major city.'    full_ids = tokenizer(full_text, return_tensors='pt')['input_ids']        with torch.no_grad():        logits = model(full_ids).logits        # Log-probs for each generated token    log_probs = torch.log_softmax(logits, dim=-1)    n_prompt = prompt_ids.shape[1]        generated_tokens = full_ids[0, n_prompt:]    generated_logps = []    for i, token_id in enumerate(generated_tokens):        pos = n_prompt + i - 1  # logit at position t predicts token t+1        lp = log_probs[0, pos, token_id].item()        tok_text = tokenizer.decode([token_id])        generated_logps.append((tok_text, lp))        print(f'  Token "{tok_text}": log-prob = {lp:.3f}')        total_seq_logp = sum(lp for _, lp in generated_logps)    print(f'\nSequence log-prob (sum): {total_seq_logp:.3f}')    print('This is log π(response | prompt) used in DPO and RLHF')except Exception as e:    print(f'Transformers demo skipped: {e}\n(Set up a model to see token-level log-probs)')

## ✅ Key Takeaways1. Each token is an MDP action drawn from a distribution over ~100k vocabulary items2. The sequence log-prob = sum of per-token log-probs (factorisation of the joint)3. RLHF reward is sparse: only at the final EOS token (reward model score)4. DPO computes log π(y|x) - log π_ref(y|x) using exactly this token-level computation